In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
from torch.utils.data import DataLoader


In [2]:
VALIDATION_LOOP_DIR = Path(".").resolve()
REPO_DIR = VALIDATION_LOOP_DIR.parent

sys.path.insert(0, str((REPO_DIR / "training_loop").resolve()))
sys.path.insert(0, str(REPO_DIR.resolve()))

DATASET_DIR = REPO_DIR.parent / "dataset"
TRAIN_DIR = DATASET_DIR / "Dataset_train"
VAL_DIR = DATASET_DIR / "Dataset_validation"

TRAIN_CSV = REPO_DIR / "tags" / "train.csv"
VAL_CSV = REPO_DIR / "tags" / "validation.csv"
OUTPUT_DIR = REPO_DIR / "output" / "history_2_5_d_simple_cnn"

label_columns = ["ICH"]
image_size = 224
num_windows = 128
window_size = 3

epoch = 10
batch_size_train = 4
batch_size_val = 4
num_workers = 4


In [3]:
from dataset_2_5d import CTWindowsDataset


class ConvBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p) if dropout_p > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)


class ResidualBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm2d(out_channels)
        self.dropout = nn.Dropout2d(p=dropout_p) if dropout_p > 0 else nn.Identity()

        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.InstanceNorm2d(out_channels),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)
        out = F.relu(self.norm1(self.conv1(x)), inplace=True)
        out = self.dropout(out)
        out = self.norm2(self.conv2(out))
        out = F.relu(out + identity, inplace=True)
        return out


class SliceEncoder2D(nn.Module):
    def __init__(self, in_channels=3, embedding_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock2D(in_channels, 16, stride=2, dropout_p=0.1),
            ResidualBlock2D(16, 16, stride=1, dropout_p=0.1),
            ResidualBlock2D(16, 32, stride=2, dropout_p=0.1),
            ResidualBlock2D(32, 32, stride=1, dropout_p=0.1),
            ResidualBlock2D(32, 64, stride=2, dropout_p=0.1),
            ResidualBlock2D(64, 64, stride=1, dropout_p=0.1),
            ResidualBlock2D(64, 128, stride=2, dropout_p=0.1),
            ResidualBlock2D(128, 128, stride=1, dropout_p=0.1),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, embedding_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.encoder(x)
        return self.head(x)


class AttentionPooling(nn.Module):
    def __init__(self, embedding_dim=256, hidden_dim=128, dropout_p=0.3):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, embeddings):
        weights = torch.softmax(self.attention(embeddings), dim=1)
        pooled = torch.sum(weights * embeddings, dim=1)
        return pooled, weights.squeeze(-1)


class Simple2_5DClassifier(nn.Module):
    def __init__(self, in_channels=3, embedding_dim=256, num_classes=1):
        super().__init__()
        self.encoder = SliceEncoder2D(in_channels=in_channels, embedding_dim=embedding_dim)
        self.pooling = AttentionPooling(embedding_dim=embedding_dim, hidden_dim=128, dropout_p=0.3)
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, windows):
        batch_size, num_windows, channels, height, width = windows.shape
        x = windows.view(batch_size * num_windows, channels, height, width)
        embeddings = self.encoder(x)
        embeddings = embeddings.view(batch_size, num_windows, -1)
        pooled, attention_weights = self.pooling(embeddings)
        logits = self.classifier(pooled)
        return logits, attention_weights


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_dataset = CTWindowsDataset(
    table_path=TRAIN_CSV,
    images_dir=TRAIN_DIR,
    label_columns=label_columns,
    image_size=image_size,
    num_windows=num_windows,
)

val_dataset = CTWindowsDataset(
    table_path=VAL_CSV,
    images_dir=VAL_DIR,
    label_columns=label_columns,
    image_size=image_size,
    num_windows=num_windows,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size_val,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)

model = Simple2_5DClassifier(in_channels=window_size, embedding_dim=256, num_classes=len(label_columns)).to(device)
checkpoint = torch.load(OUTPUT_DIR / f"checkpoint_epoch_{epoch}.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


Using device: cpu


Simple2_5DClassifier(
  (encoder): SliceEncoder2D(
    (encoder): Sequential(
      (0): ConvBlock2D(
        (block): Sequential(
          (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
          (2): ReLU(inplace=True)
          (3): Dropout2d(p=0.1, inplace=False)
        )
      )
      (1): ResidualBlock2D(
        (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (norm1): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (norm2): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (dropout): Dropout2d(p=0.1, inplace=False)
        (skip): Identity()
      )
      (2): ResidualBlock2D(
        (conv1): Conv2d(16, 32, kernel_siz

In [5]:
def collect_split_predictions(model, loader, device):
    targets = []
    scores = []

    with torch.no_grad():
        for windows, labels in loader:
            windows = windows.to(device, non_blocking=True)
            logits, _ = model(windows)
            probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)

            scores.append(probs)
            targets.append(labels.numpy().reshape(-1))

    y_true = np.concatenate(targets, axis=0)
    y_score = np.concatenate(scores, axis=0)
    return y_true, y_score


train_y_true, train_y_score = collect_split_predictions(
    model=model,
    loader=train_loader,
    device=device,
)

val_y_true, val_y_score = collect_split_predictions(
    model=model,
    loader=val_loader,
    device=device,
)


In [6]:
def plot_preliminary_plots(y_true, y_score, dataset_name):
    # Добавляем width и height в гистограмму
    fig_hist = px.histogram(x=y_score, color=y_true.astype(str), nbins=50,
                            labels={'color': 'True Labels', 'x': 'Score'},
                            title=f'{dataset_name}: Histogram of Scores',
                            width=800,   # ← ДОБАВИТЬ
                            height=500)  # ← ДОБАВИТЬ
    fig_hist.show()

    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    df_thresh = pd.DataFrame({'FPR': fpr, 'TPR': tpr}, index=thresholds)
    
    # Добавляем width и height в график TPR/FPR
    fig_thresh = px.line(df_thresh, 
                         title=f'{dataset_name}: TPR and FPR at every threshold',
                         width=800,    # ← ДОБАВИТЬ
                         height=500)   # ← ДОБАВИТЬ
    fig_thresh.update_yaxes(scaleanchor="x", scaleratio=1)
    fig_thresh.update_xaxes(range=[0, 1], constrain='domain')
    fig_thresh.show()

def plot_roc_curve(y_true, y_score, dataset_name):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    
    # Добавляем width и height в ROC кривую
    fig = px.area(x=fpr, y=tpr, 
                  title=f'{dataset_name}: ROC Curve (AUC={roc_auc:.3f})',
                  labels={'x': 'False Positive Rate', 'y': 'True Positive Rate'},
                  width=800,    # ← ДОБАВИТЬ
                  height=500)   # ← ДОБАВИТЬ
    fig.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0, y1=1)
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()

def plot_pr_curve(y_true, y_score, dataset_name):
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(recall, precision)
    
    # Добавляем width и height в PR кривую
    fig = px.area(x=recall, y=precision, 
                  title=f'{dataset_name}: Precision-Recall Curve (AUC={pr_auc:.3f})',
                  labels={'x': 'Recall', 'y': 'Precision'},
                  width=800,    # ← ДОБАВИТЬ
                  height=500)   # ← ДОБАВИТЬ
    fig.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=1, y1=0)
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()


In [7]:
for y_true, y_score, name in [(train_y_true, train_y_score, 'Train'),
                               (val_y_true, val_y_score, 'Validation')]:
    plot_preliminary_plots(y_true, y_score, name)
    plot_roc_curve(y_true, y_score, name)
    plot_pr_curve(y_true, y_score, name)


In [8]:
import pandas as pd

pd.Series(train_y_score).to_csv(OUTPUT_DIR / 'train_predictions.csv', index=False, header=True)
pd.Series(val_y_score).to_csv(OUTPUT_DIR / 'validation_predictions.csv', index=False, header=True)
